In [1]:
# ===== C1 clone + clock =====
import time, subprocess
NB_START=time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True); print("cloned")

cloned


In [2]:
%%writefile /kaggle/temp/FreeFine/ip_ff.py
# ip_ff.py — FreeFine-ID (B2): IP-Adapter image-identity conditioning for FreeFine's
# custom attention, env-flag gated. Adds a decoupled image cross-attention to the
# EDIT-conditional branch of modulate_local_cross_attn ONLY. TCA (self-attn) untouched.
#
# Design (verified against src/utils/attention.py @ commit 4c9fdb9):
#   - FreeFine cross-attn path: register_attention_control -> (is_cross & controller.local_edit)
#     -> controller.modulate_local_cross_attn(q,k,v,is_cross,place_in_unet)
#   - inside it, after batch_to_head_dim the 4 chunks are [uncon_edit, uncon_ref, con_edit, con_ref];
#     final output blends con_edit/uncon_edit by local_region (target mask, downsampled).
#   - We add  ip_scale * local_region * attn(query_chunk, to_k_ip(ip_tokens), to_v_ip(ip_tokens))
#     to con_edit (and uncon_edit) BEFORE the blend, so the IP signal is auto-masked to the object.
#
# Weight provenance: h94/IP-Adapter  ip-adapter_sd15.bin  (image_proj + ip_adapter)
#                    image encoder: CLIP ViT-H/14 (models/image_encoder), image_embeds dim 1024.

import os, math, torch
import torch.nn as nn
import numpy as np
from PIL import Image


# ---------------------------------------------------------------- image projection
class ImageProjModel(nn.Module):
    """ip-adapter_sd15: 1024-dim CLIP image_embeds -> 4 context tokens x 768."""
    def __init__(self, cross_attention_dim=768, clip_embeddings_dim=1024, clip_extra_context_tokens=4):
        super().__init__()
        self.clip_extra_context_tokens = clip_extra_context_tokens
        self.cross_attention_dim = cross_attention_dim
        self.proj = nn.Linear(clip_embeddings_dim, clip_extra_context_tokens * cross_attention_dim)
        self.norm = nn.LayerNorm(cross_attention_dim)

    def forward(self, image_embeds):
        x = self.proj(image_embeds).reshape(-1, self.clip_extra_context_tokens, self.cross_attention_dim)
        return self.norm(x)


# ---------------------------------------------------------------- loader
def load_ip(model, ckpt_path, image_encoder_path, device, dtype=torch.float32, num_tokens=4,
            image_encoder_subfolder="models/image_encoder", enc_device="cpu"):
    """Attach ip_to_k / ip_to_v to every cross-attn (attn2) module of model.unet, load the
    image projector + CLIP image encoder. Returns a dict of handles for encoding crops.
    Asserts the 16-layer match so a wrong checkpoint fails loudly, not silently.

    The big CLIP ViT-H encoder + projector live on `enc_device` (default CPU) so they do
    NOT compete with FreeFine's memory-heavy self-attention on the GPU; only the tiny
    ip_to_k/ip_to_v projections sit on `device`. encode_tokens() returns tokens already
    moved to `device`, so callers need no change."""
    from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

    sd = torch.load(ckpt_path, map_location="cpu")
    assert "image_proj" in sd and "ip_adapter" in sd, "unexpected IP-Adapter checkpoint format"

    image_proj = ImageProjModel(768, 1024, num_tokens)
    image_proj.load_state_dict(sd["image_proj"])
    image_proj = image_proj.to(enc_device, dtype=torch.float32).eval()

    # cross-attn modules in UNet DFS order == diffusers canonical order == IP weight order
    attn2 = [(n, m) for n, m in model.unet.named_modules() if n.endswith("attn2")]
    ip = sd["ip_adapter"]
    prefixes = sorted({int(k.split(".")[0]) for k in ip}, key=int)
    assert len(attn2) == len(prefixes), \
        f"cross-attn count {len(attn2)} != IP layer count {len(prefixes)} (wrong base model?)"

    for (name, mod), p in zip(attn2, prefixes):
        inner = mod.to_q.out_features
        kw = ip[f"{p}.to_k_ip.weight"]
        vw = ip[f"{p}.to_v_ip.weight"]
        assert kw.shape == (inner, 768) and vw.shape == (inner, 768), \
            f"{name}: ip weight shape {tuple(kw.shape)} != ({inner},768)"
        k = nn.Linear(768, inner, bias=False)
        v = nn.Linear(768, inner, bias=False)
        k.weight.data.copy_(kw); v.weight.data.copy_(vw)
        mod.ip_to_k = k.to(device, dtype=dtype)
        mod.ip_to_v = v.to(device, dtype=dtype)

    enc = CLIPVisionModelWithProjection.from_pretrained(
        image_encoder_path, subfolder=image_encoder_subfolder).to(enc_device, dtype=torch.float32).eval()
    proc = CLIPImageProcessor()
    print(f"[B2] IP-Adapter loaded: {len(attn2)} cross-attn layers, {num_tokens} tokens "
          f"(encoder on {enc_device}, projections on {device})")
    return {"image_proj": image_proj, "encoder": enc, "processor": proc,
            "enc_device": enc_device, "token_device": device, "dtype": dtype, "num_tokens": num_tokens}


# ---------------------------------------------------------------- crop + encode
def object_crop(ori_img, ori_mask, pad=0.10, gray_bg=True):
    """Bounding-box crop of the source object; optional gray background so identity
    (not surroundings) drives the embedding. ori_img HxWx3 uint8, ori_mask HxW{0,1/255}."""
    m = (np.asarray(ori_mask) > 0)
    if m.ndim == 3: m = m[..., 0]
    ys, xs = np.where(m)
    if len(ys) == 0:
        return Image.fromarray(np.asarray(ori_img).astype(np.uint8))
    H, W = m.shape
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    py, px = int((y1 - y0) * pad), int((x1 - x0) * pad)
    y0, y1 = max(0, y0 - py), min(H, y1 + py + 1)
    x0, x1 = max(0, x0 - px), min(W, x1 + px + 1)
    img = np.asarray(ori_img).astype(np.uint8).copy()
    if gray_bg:
        img[~m] = 127
    return Image.fromarray(img[y0:y1, x0:x1])


@torch.no_grad()
def encode_tokens(handles, pil_crop):
    """PIL crop -> (1, num_tokens, 768) image-context tokens, on token_device (the GPU).
    Encoder runs on enc_device (CPU) to keep GPU memory free; only the tiny token tensor
    moves to the GPU."""
    px = handles["processor"](images=pil_crop, return_tensors="pt").pixel_values
    px = px.to(handles["enc_device"], dtype=torch.float32)
    emb = handles["encoder"](px).image_embeds            # (1, 1024) on enc_device
    tok = handles["image_proj"](emb)                     # (1, num_tokens, 768) on enc_device
    return tok.to(handles["token_device"], dtype=handles["dtype"])


# ---------------------------------------------------------------- attention hook
def ip_cross_branch(controller, attn_module, query_hb, local_region_flat):
    """Compute the IP additive term for the edit branch.
    query_hb: (4*heads, seq, head_dim) in chunk order [uncon_edit, uncon_ref, con_edit, con_ref].
    Returns (add_uncon_edit, add_con_edit) each (seq, inner), already mask-scaled by local_region.
    Called from the patched modulate_local_cross_attn; no-op handled by caller."""
    H = controller.heads
    tok = controller.ip_tokens.to(query_hb.dtype)            # (1, ntok, 768)
    ik = attn_module.ip_to_k(tok)                            # (1, ntok, inner)
    iv = attn_module.ip_to_v(tok)
    ik = controller.head_to_batch_dim(ik)                    # (H, ntok, head_dim)
    iv = controller.head_to_batch_dim(iv)

    def one(chunk_start):
        q = query_hb[chunk_start * H:(chunk_start + 1) * H]  # (H, seq, head_dim)
        out = controller.mask_attention(q, ik, iv, None)     # (H, seq, head_dim)
        out = controller.batch_to_head_dim(out)[0]           # (seq, inner)
        return controller.ip_scale * local_region_flat[:, None] * out

    add_uncon = one(0)   # uncon_edit
    add_con = one(2)     # con_edit
    return add_uncon, add_con

Writing /kaggle/temp/FreeFine/ip_ff.py


In [3]:
%%bash
# ===== C2 freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 77.2 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.34s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 1.71s
 Downloaded torchvision
 Downloaded triton
 Downloaded pillow
 Downloaded networkx
 Downloaded numpy
 Downloaded sympy
 Downloaded torch
Prepared 18 packages in 39.15s
Installed 18 packages in 269ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

In [4]:
%%bash
# ===== C3 metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK"

metric_env OK


Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 1.05s
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded pillow
 Downloaded nvidia-curand-cu12
 Downloaded networkx
 Downloaded numpy
 Downloaded nvidia-cusolver-cu12
 Downloaded triton
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 46.94s
Installed 27 packages in 262ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4

In [5]:
# ===== C4 patch metrics (args.3d, SD-2.1 mirror, SEEDED MD) + model.py START_LAYER fix =====
import pathlib, re
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [mr/"MD"/"mean_distance.py", mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=mr/"MD"/"mean_distance.py"; s=md.read_text()
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st; _st.manual_seed(42); _st.cuda.manual_seed_all(42)",1); md.write_text(s)
mm=pathlib.Path("/kaggle/temp/FreeFine/src/demo/model.py"); g=mm.read_text()
if not re.search(r'^\s*import os\b', g, re.M): g="import os\n"+g
assert "list(range(10, 16))" in g, "layer_idx hardcode missing"
n=g.count("list(range(10, 16))")
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
mm.write_text(g); print(f"patched metrics + model.py start_layer ({n} sites)")

patched metrics + model.py start_layer (5 sites)


In [6]:
# ===== C5 parametrize inference script (prompt + guidance + start_step + paths + ori_mask fix) =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"','pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src, "ori_mask/obj_label block mismatch"; src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src)
assert all(x in src for x in ["FF_USE_PROMPT","FF_GUIDANCE","FF_START_STEP"]), "parametrize failed"
print("parametrized: prompt + guidance + start_step (start_layer via model.py)")

parametrized: prompt + guidance + start_step (start_layer via model.py)


In [7]:
# ===== C5b B1 patches: AA-Warp + Adaptive-ES + save-coarse (on freefine_sweep_2d.py) =====
SW = "/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"
src = open(SW).read()
def patch(t, a, r, name):
    assert a in t, f"ANCHOR NOT FOUND: {name}"
    assert r not in t, f"ALREADY PATCHED: {name}"
    return t.replace(a, r, 1)

# P1: AA-Warp (supersampled Lanczos), gate FF_AA_WARP=1; mask warp untouched (stays NEAREST)
P1_A = "    transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))"
P1_R = """    if os.environ.get('FF_AA_WARP', '0') == '1':
        _S = 2
        _M = rotation_matrix.copy()
        _M[:, 2] *= _S
        _src_up = cv2.resize(src_img, (width * _S, height * _S), interpolation=cv2.INTER_LANCZOS4)
        _img_up = cv2.warpAffine(_src_up, _M, (width * _S, height * _S), flags=cv2.INTER_LANCZOS4)
        transformed_image = cv2.resize(_img_up, (width, height), interpolation=cv2.INTER_AREA)
    else:
        transformed_image = cv2.warpAffine(src_img, rotation_matrix, (width, height))"""
src = patch(src, P1_A, P1_R, "P1 AA-warp")

# P2a: difficulty LUT at module level (loads only when FF_META_CSV is set)
P2A_A = "def main(dst_base):"
P2A_R = """FF_DIFF_LUT = {}
_meta_csv = os.environ.get('FF_META_CSV', '')
if _meta_csv:
    import csv as _csv
    with open(_meta_csv) as _f:
        for _row in _csv.DictReader(_f):
            FF_DIFF_LUT[(str(_row['da_n']), str(_row['ins_id']), str(_row['case_id']))] = _row['difficulty']
    print(f'[B1] difficulty LUT loaded: {len(FF_DIFF_LUT)} cases', flush=True)
    assert len(FF_DIFF_LUT) == 5677, 'difficulty LUT incomplete'

def main(dst_base):"""
src = patch(src, P2A_A, P2A_R, "P2a LUT")

# P2b: per-case end_scale
P2B_A = "        edit_param = case['edit_param']"
P2B_R = """        edit_param = case['edit_param']
        if os.environ.get('FF_ES_MODE', 'fixed') == 'adaptive':
            _diff = FF_DIFF_LUT.get((str(da_n), str(ins_id), str(edit_ins)), 'medium')
            _ff_end_scale = {'easy': 0.0, 'medium': 0.25, 'hard': 0.5}[_diff]
        else:
            _ff_end_scale = float(os.environ.get('FF_END_SCALE', '0.0'))"""
src = patch(src, P2B_A, P2B_R, "P2b adaptive ES")

src = patch(src, '            "end_scale": 0.0,', '            "end_scale": _ff_end_scale,', "P2c wire-in")

# P3: save the (possibly AA) coarse so WRAP_E can use the right reference
P3_A = "        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)"
P3_R = """        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)
        if os.environ.get('FF_SAVE_COARSE', '0') == '1':
            _cdir = os.path.join(os.environ.get('FF_COARSE_DIR', '/kaggle/working/b1_coarse'), str(da_n), str(ins_id))
            os.makedirs(_cdir, exist_ok=True)
            Image.fromarray(coarse_input.astype(np.uint8)).save(os.path.join(_cdir, f"{edit_ins}.png"))"""
src = patch(src, P3_A, P3_R, "P3 save coarse")

open(SW, "w").write(src)
print("[B1] all 5 patches applied to freefine_sweep_2d.py")

[B1] all 5 patches applied to freefine_sweep_2d.py


In [8]:
# ===== IP-B: patch attention.py (cross-attn hook) + freefine_sweep_2d.py (load + per-case) =====
A="/kaggle/temp/FreeFine/src/utils/attention.py"; a=open(A).read()
def P(t,old,new,name):
    assert old in t, f"ANCHOR NOT FOUND: {name}"
    assert new not in t, f"ALREADY PATCHED: {name}"
    return t.replace(old,new,1)

# 1: pass the Attention module into the cross-attn modulator (so it can reach ip_to_k/ip_to_v)
a=P(a,
"                hidden_states = controller.modulate_local_cross_attn(query,key,value,is_cross, place_in_unet)",
"                hidden_states = controller.modulate_local_cross_attn(query,key,value,is_cross, place_in_unet, attn_module=self)",
"call-site")

# 2a: signature gains attn_module
a=P(a,
"    def modulate_local_cross_attn(self,query,key,value,is_cross, place_in_unet):",
"    def modulate_local_cross_attn(self,query,key,value,is_cross, place_in_unet, attn_module=None):",
"signature")

# 2b: add the IP term right after the (unique) 4-chunk unpack, before the blend
a=P(a,
"        _, L1, L2 = hidden_states.shape #4,4096,320 [uncon_edit,uncon_ref,con_edit,con_ref]\n        uncon_edit,uncon_ref,con_edit,_ = hidden_states\n",
"        _, L1, L2 = hidden_states.shape #4,4096,320 [uncon_edit,uncon_ref,con_edit,con_ref]\n        uncon_edit,uncon_ref,con_edit,_ = hidden_states\n"
"        if getattr(self,'ip_tokens',None) is not None and attn_module is not None and getattr(attn_module,'ip_to_k',None) is not None:\n"
"            import ip_ff as _ipff\n"
"            _au,_ac = _ipff.ip_cross_branch(self, attn_module, query, local_region)\n"
"            uncon_edit = uncon_edit + _au; con_edit = con_edit + _ac\n",
"ip-add")
open(A,"w").write(a); print("attention.py patched")

S="/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"; s=open(S).read()
# 3a: load IP handles once (FF_IP=1), after model/attention setup
s=P(s,
"    model.enable_xformers_memory_efficient_attention()",
"    model.enable_xformers_memory_efficient_attention()\n"
"    if os.environ.get('FF_IP')=='1':\n"
"        import ip_ff\n"
"        from huggingface_hub import hf_hub_download\n"
"        _ckpt=hf_hub_download('h94/IP-Adapter','models/ip-adapter_sd15.bin')\n"
"        model._ip=ip_ff.load_ip(model,_ckpt,'h94/IP-Adapter',device)\n",
"ip-load")
# 3b: set per-case image tokens just before generation
s=P(s,
"        generated_results = model.FreeFine_generation(**params)",
"        if os.environ.get('FF_IP')=='1':\n"
"            import ip_ff\n"
"            model.controller.ip_tokens=ip_ff.encode_tokens(model._ip, ip_ff.object_crop(ori_img, ori_mask))\n"
"            model.controller.ip_scale=float(os.environ.get('FF_IP_SCALE','0.6'))\n"
"        else:\n"
"            model.controller.ip_tokens=None\n"
"        generated_results = model.FreeFine_generation(**params)",
"ip-percase")
open(S,"w").write(s); print("freefine_sweep_2d.py patched")

attention.py patched
freefine_sweep_2d.py patched


In [9]:
# ===== C6 data + balanced-200 + coarse + reuse saved start_step variants =====
import os, glob, json, csv, random, shutil
from collections import defaultdict, Counter
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)
CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True) if os.path.isdir(f"{c}/source_img"))
COARSE=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png",recursive=True)[0].split("/coarse_img/")[0]+"/coarse_img"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
ANNs=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]; META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
VS=glob.glob("/kaggle/input/**/phase2/variants",recursive=True); VS=VS[0] if VS else None
for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(sc,d)
shutil.copy(ANNs,f"{GEO}/annotation_2d.json"); ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META)) if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]
random.seed(42); cells=defaultdict(list)
for r in meta: cells[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(cells); per=200//len(keys); picked=[]
for k in keys:
    pool=cells[k][:]; random.shuffle(pool); picked+=pool[:per]
ch={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in ch]; random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]; json.dump(picked,open(f"{GEO}/subset_meta.json","w"))
print("subset",len(picked),dict(Counter(r["edit_type"] for r in picked)))
gsub={}
for r in picked:
    d,i,e=r["da_n"],r["ins_id"],r["case_id"]; lf=dict(ann[d]["instances"][i][e])
    lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"]); lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
    gsub.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
json.dump(gsub,open(f"{GEO}/gen_subset.json","w"))
os.makedirs(f"{GEO}/gen_eval",exist_ok=True); sets={"baseline":GENBASE}
if VS:
    for tag in ["ss25","ss30","ss40","ss45"]:
        if glob.glob(f"{VS}/{tag}/**/*.png",recursive=True): sets[tag]=f"{VS}/{tag}"
for nm,sc in sets.items():
    d=f"{GEO}/gen_eval/{nm}"
    if os.path.islink(d): os.remove(d)
    os.symlink(sc,d)
print("reused eval sets:",list(sets))

subset 200 {'move': 67, 'resize': 67, 'rotate': 66}
reused eval sets: ['baseline', 'ss25', 'ss30', 'ss40', 'ss45']


In [10]:
# ===== C7 validation gate (default params must reproduce baseline) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5:break
        if n>=5:break
    if n>=5:break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation 0 images")
diffs=[float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{os.path.relpath(n,'/kaggle/temp/val5')}").convert("RGB").resize(Image.open(n).size),float)).mean()) for n in imgs]
print("validation mean|Δ|:",[round(x,3) for x in diffs]); assert max(diffs)<1.0,"NOT REPRODUCING"; print("✓ validation passed")

validation mean|Δ|: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ validation passed


In [11]:
# ===== C8 generate NEW variants: start_layer (fixed) + prompt-enabled CFG =====
import os, time, glob, subprocess, socket, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; OUT="/kaggle/working/b2/variants"; os.makedirs(OUT,exist_ok=True)
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
GEN_DEADLINE=NB_START+6.5*3600
import glob as _g
META=_g.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
NEW=[("ip50", {"FF_IP":1,"FF_IP_SCALE":0.5}),
     ("ip80", {"FF_IP":1,"FF_IP_SCALE":0.8}),
     ("ip110",{"FF_IP":1,"FF_IP_SCALE":1.1}),
     ("ip140",{"FF_IP":1,"FF_IP_SCALE":1.4})]
def gen2(o,**kw):
    os.makedirs(o,exist_ok=True); env=os.environ.copy()
    env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/gen_subset.json","FF_OUT_DIR":o})
    env.update({k:str(v) for k,v in kw.items()})
    s=socket.socket();s.bind(("",0));p=s.getsockname()[1];s.close()
    return subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=2","--master-port",str(p),"freefine_sweep_2d.py"],cwd="/kaggle/temp/FreeFine/evaluation/FreeFine",env=env,stdout=open(o+"/log.txt","w"),stderr=subprocess.STDOUT)
def pdiff(tag):
    ds=[]
    for n in sorted(glob.glob(f"{OUT}/{tag}/**/*.png",recursive=True))[:8]:
        rel=os.path.relpath(n,f"{OUT}/{tag}")
        ds.append(round(float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{rel}").convert("RGB").resize(Image.open(n).size),float)).mean()),2))
    return ds
done=[]
for tag,kw in NEW:
    if time.time()>GEN_DEADLINE: print("GEN_DEADLINE skip",tag,flush=True); break
    o=f"{OUT}/{tag}"; t=time.time()
    try:
        r=gen2(o,**kw); k=len(glob.glob(o+"/**/*.png",recursive=True))
        print(f"[{int((time.time()-NB_START)/60)}m] {tag}: rc={r.returncode} {int(time.time()-t)}s imgs={k} Δvs_baseline={pdiff(tag)}",flush=True)
        if k>0: done.append(tag)
        if r.returncode!=0: print("  tail:",open(o+'/log.txt').read()[-900:])
    except Exception as ex: print(f"{tag} FAILED:{ex}",flush=True)
print("new variants:",done,flush=True)

[94m] ip50: rc=0 5027s imgs=200 Δvs_baseline=[0.16, 0.18, 0.29, 0.29, 0.38, 0.1, 0.09, 0.19]
[179m] ip80: rc=0 5061s imgs=200 Δvs_baseline=[0.25, 0.26, 0.41, 0.38, 0.57, 0.15, 0.13, 0.29]
[263m] ip110: rc=0 5067s imgs=200 Δvs_baseline=[0.33, 0.32, 0.5, 0.45, 0.74, 0.22, 0.17, 0.39]
[347m] ip140: rc=0 5056s imgs=200 Δvs_baseline=[0.41, 0.37, 0.57, 0.54, 0.89, 0.32, 0.21, 0.48]
new variants: ['ip50', 'ip80', 'ip110', 'ip140']


In [12]:
# ===== C9 eval — B2 (baseline + IP scales) =====
import os, json, re, glob, subprocess, time, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"; PY="/kaggle/temp/metric_env/bin/python"
EVAL_DEADLINE=NB_START+10.5*3600
ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))

# link the B2 variant outputs into gen_eval
for tag in ["ip50","ip80","ip110","ip140"]:
    p=f"/kaggle/working/b2/variants/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True):
        d=f"{GEO}/gen_eval/{tag}"
        if os.path.islink(d): os.remove(d)
        os.symlink(p,d)
sets=[s for s in ["baseline","ip50","ip80","ip110","ip140"] if os.path.exists(f"{GEO}/gen_eval/{s}")]
print("eval sets:",sets)

def mem(pred): return [(r["da_n"],r["ins_id"],r["case_id"]) for r in picked if pred(r)]
groups={"rotate_hard":(mem(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),"000110100"),
        "resize_hard":(mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),"000110100"),
        "move_all":(mem(lambda r:r["edit_type"]=="move"),"000110000"),
        "all_200":(mem(lambda r:True),"100110011")}

def wrap_e(ids,gd):
    tot=0.0;n=0
    for d,i,e in ids:
        cp=f"{GEO}/Geo-Bench-2D/coarse_img/{d}/{i}/{e}.png"; gp=f"{gd}/{d}/{i}/{e}.png"; tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"
        if not(os.path.exists(cp) and os.path.exists(gp) and os.path.exists(tp)): continue
        C=np.array(Image.open(cp).convert("RGB"),float)/255; G=np.array(Image.open(gp).convert("RGB"),float)/255; T=np.array(Image.open(tp).convert("L"),float)/255
        if G.shape[:2]!=C.shape[:2]: G=np.array(Image.fromarray((G*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        if T.shape[:2]!=C.shape[:2]: T=np.array(Image.fromarray((T*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        m=np.repeat(T[...,None],3,axis=2); su=m.sum()
        if su<=0: continue
        tot+=float(np.sum(np.abs(C*m-G*m))/su); n+=1
    return round(tot/n,4) if n else None

def manifest(setn,ids):
    o={}; b=f"{GEO}/gen_eval/{setn}"
    for d,i,e in ids:
        if not os.path.exists(f"{b}/{d}/{i}/{e}.png"): continue
        lf=dict(ann[d]["instances"][i][e]); lf["gen_img_path"]=f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    pth=f"{GEO}/m_{setn}_{len(ids)}.json"; json.dump(o,open(pth,"w")); return pth

def runm(manp,task):
    env=os.environ.copy(); env.update({"MPLBACKEND":"Agg","HF_HOME":"/kaggle/temp/hf","TORCH_HOME":"/kaggle/temp/torch","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
    out=subprocess.run([PY,"main.py","--path",manp,"--use_relative_path","--base_dir",GEO,"--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",task,"--level","0"],cwd=MET,env=env,capture_output=True,text=True)
    t=out.stdout+out.stderr; v={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","MD"]:
        m=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",t)
        if m: v[k]=round(float(m[-1]),4)
    return v

results={}; stop=False
for setn in sets:
    if stop: break
    results[setn]={}
    for g,(ids,task) in groups.items():
        if time.time()>EVAL_DEADLINE: print("EVAL_DEADLINE",flush=True); stop=True; break
        if not ids: continue
        v=runm(manifest(setn,ids),task); v["WRAP_E"]=wrap_e(ids,f"{GEO}/gen_eval/{setn}"); v["n"]=len(ids)
        results[setn][g]=v; print(f"{setn:8s} {g:12s} -> {v}",flush=True)
    json.dump(results,open("/kaggle/working/b2_full.json","w"),indent=2)
print("eval done")

eval sets: ['baseline', 'ip50', 'ip80', 'ip110', 'ip140']
baseline rotate_hard  -> {'SUBC': 0.8523, 'BGC': 0.9664, 'MD': 14.0096, 'WRAP_E': 0.0416, 'n': 22}
baseline resize_hard  -> {'SUBC': 0.8391, 'BGC': 0.9631, 'MD': 19.4713, 'WRAP_E': 0.0585, 'n': 23}
baseline move_all     -> {'SUBC': 0.959, 'BGC': 0.9639, 'WRAP_E': 0.0495, 'n': 67}
baseline all_200      -> {'FID_DINO': 1636.8248, 'FID_KD': 0.1278, 'FID': 132.279, 'SUBC': 0.9154, 'BGC': 0.9657, 'WRAP_E': 0.0488, 'n': 200}
ip50     rotate_hard  -> {'SUBC': 0.8523, 'BGC': 0.9669, 'MD': 14.6752, 'WRAP_E': 0.0416, 'n': 22}
ip50     resize_hard  -> {'SUBC': 0.8396, 'BGC': 0.9629, 'MD': 21.4861, 'WRAP_E': 0.0584, 'n': 23}
ip50     move_all     -> {'SUBC': 0.959, 'BGC': 0.9638, 'WRAP_E': 0.0495, 'n': 67}
ip50     all_200      -> {'FID_DINO': 1637.4637, 'FID_KD': 0.1321, 'FID': 132.0828, 'SUBC': 0.9154, 'BGC': 0.9656, 'WRAP_E': 0.0488, 'n': 200}
ip80     rotate_hard  -> {'SUBC': 0.8518, 'BGC': 0.9668, 'MD': 14.8623, 'WRAP_E': 0.0416, 'n': 

In [13]:
# ===== C10 summary — B2 =====
import json
R=json.load(open("/kaggle/working/b2_full.json"))
order=["baseline","ip50","ip80","ip110","ip140"]
for g in ["rotate_hard","resize_hard","move_all","all_200"]:
    print(f"\n=== {g} ===")
    cols=["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"]
    print("set      "+"".join(f"{c:>10}" for c in cols))
    for s in order:
        if s in R and g in R[s]:
            d=R[s][g]; print(f"{s:8s} "+"".join(f"{str(d.get(c,'-')):>10}" for c in cols))
print("\nB2: IP-Adapter identity conditioning, scale 0.5/0.8/1.1/1.4. baseline = FreeFine repro.")
print("saved -> /kaggle/working/b2_full.json")


=== rotate_hard ===
set            SUBC       BGC    WRAP_E        MD       FID  FID_DINO    FID_KD
baseline     0.8523    0.9664    0.0416   14.0096         -         -         -
ip50         0.8523    0.9669    0.0416   14.6752         -         -         -
ip80         0.8518    0.9668    0.0416   14.8623         -         -         -
ip110        0.8512    0.9668    0.0417   15.4672         -         -         -
ip140        0.8502    0.9667    0.0417    15.228         -         -         -

=== resize_hard ===
set            SUBC       BGC    WRAP_E        MD       FID  FID_DINO    FID_KD
baseline     0.8391    0.9631    0.0585   19.4713         -         -         -
ip50         0.8396    0.9629    0.0584   21.4861         -         -         -
ip80         0.8394    0.9627    0.0584   18.7526         -         -         -
ip110        0.8393    0.9627    0.0585   19.4285         -         -         -
ip140         0.839    0.9624    0.0586   21.5396         -         -         